# 35 — Privacy: clipping and noise in DP-SGD

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/035-privacy-and-threat-model.ipynb)

A model that memorises one training example leaks that example. Differential privacy caps each example's influence by clipping its gradient, then hides what is left under Gaussian noise. This notebook builds that update from one gradient, trains with it, and prices the privacy in accuracy.

## 1. The numbers: one gradient, clipped and noised

Start with a single example. Its gradient is a two-number vector, one entry for each model weight, and its length says how loudly that example speaks. Clipping shortens the vector to at most `C`; noise is then added to whatever survives.

In [ ]:
import numpy as np                                   # the array library, preinstalled on Colab

x, y = 2.5, 0.0                                      # one example: feature 2.5, label class 0
w, b = 0.3, -0.1                                     # the current model weights
z = w * x + b                                        # the score before the sigmoid
p = 1.0 / (1.0 + np.exp(-z))                         # predicted probability of class 1
g = (p - y) * np.array([x, 1.0])                     # gradient with respect to w and b
C = 1.0                                              # the clipping norm
length = float(np.linalg.norm(g))                    # length of the gradient vector
g_clip = g * min(1.0, C / length)                    # shrink it to length C at most
noise = np.random.default_rng(0).normal(0.0, 0.5 * C, 2)   # sigma = 0.5, one draw per weight

print("score:", round(float(z), 3), " probability:", round(float(p), 3))   # 0.65, 0.657
print("gradient:", np.round(g, 3), " length:", round(length, 3))           # 1.769 > C
print("clipped :", np.round(g_clip, 3), " length:", round(float(np.linalg.norm(g_clip)), 3))   # 1.0
print("noise   :", np.round(noise, 3), " sigma:", 0.5)                     # two small draws

**Line by line**

- `import numpy as np` — loads NumPy, which Colab already has.
- `x, y = 2.5, 0.0` — one example's feature and its label.
- `w, b = 0.3, -0.1` — the two numbers the model has learned so far.
- `z = ...` — the raw score, before it is turned into a probability.
- `p = ...` — the logistic function maps the score into 0 to 1.
- `g = ...` — the gradient: one entry per model weight.
- `C = 1.0` — the largest length any single example is allowed to have.
- `length = ...` — measure the gradient before clipping.
- `g_clip = ...` — scale the vector down if it is longer than `C`.
- `noise = ...` — two Gaussian draws, standard deviation `sigma` times `C`.
- `print("score"...)` — the score and the probability it produced.
- `print("gradient"...)` — the raw vector and its length, above the cap.
- `print("clipped"...)` — the shortened vector and its new length.
- `print("noise"...)` — the noise that will be added to the sum over examples.

## 2. Step by step: DP-SGD on eighty examples

Now the real loop. Each step samples four examples, clips their gradients, sums them, adds noise, and averages. Accuracy is measured on 400 held-out examples; the classes overlap, so a perfect score is impossible even without noise.

In [ ]:
import numpy as np                                   # reload the array library for this cell

rng = np.random.default_rng(0)                       # fixed seed: identical data on every run
x = np.concatenate([rng.normal(-0.5, 0.5, 40),      # 40 class-0 examples, centred left of zero
                    rng.normal(0.5, 0.5, 40)])       # 40 class-1 examples, centred right of zero
y = np.concatenate([np.zeros(40), np.ones(40)])      # the matching labels
xt = np.concatenate([rng.normal(-0.5, 0.5, 200),    # 200 held-out class-0 examples
                     rng.normal(0.5, 0.5, 200)])     # 200 held-out class-1 examples
yt = np.concatenate([np.zeros(200), np.ones(200)])   # their labels, used only for scoring
n = len(y)                                           # 80 training examples

def sigmoid(v):                                      # squash any score into 0 to 1
    return 1.0 / (1.0 + np.exp(-v))                  # the logistic function

def per_example_grads(w, b):                         # one gradient row per training example
    p = sigmoid(w * x + b)                           # the model's probability for each example
    return (p - y)[:, None] * np.stack([x, np.ones(n)], axis=1)   # (p - y) * [x, 1]

def clip(G, C):                                      # cap every row at length C
    lengths = np.linalg.norm(G, axis=1)              # length of each row
    scale = np.minimum(1.0, C / np.maximum(lengths, 1e-12))   # shrink factor, never above 1
    return G * scale[:, None]                        # apply the factor row by row

def dp_sgd(C=1.0, sigma=1.0, steps=150, lr=0.5, batch=4, seed=0):   # the training loop
    rng = np.random.default_rng(seed)                # private stream for batches and noise
    w, b = 0.0, 0.0                                  # start from the zero model
    accs = []                                        # test accuracy after every step
    for t in range(steps):                           # one noisy update per step
        idx = rng.integers(0, n, size=batch)         # a random mini-batch of four examples
        G = clip(per_example_grads(w, b)[idx], C)    # clip each example's gradient
        total = G.sum(axis=0) + rng.normal(0.0, sigma * C, 2)   # sum, then add noise
        w -= lr * total[0] / batch                   # noisy average step on w
        b -= lr * total[1] / batch                   # noisy average step on b
        pred = (w * xt + b > 0).astype(float)        # class 1 whenever the score is positive
        accs.append(float((pred == yt).mean()))      # share of the 400 test examples correct
    return accs                                      # one accuracy per step

accs = dp_sgd(C=1.0, sigma=2.0)                      # one run with visible noise
for t in range(0, len(accs), 30):                    # every thirtieth step ...
    print(f"step {t+1:3d}: accuracy {accs[t]:.2f}")  # ... print the test accuracy
print("final accuracy:", round(accs[-1], 2))         # where the run ends up

**Line by line**

- `import numpy as np` — the array library, reloaded for this cell.
- `rng = ...` — a seeded generator, so the data never changes.
- `x = np.concatenate(...)` — 40 examples centred at -0.5.
- `rng.normal(0.5, 0.5, 40)` — 40 more centred at +0.5; the spread makes them overlap.
- `y = ...` — class labels 0 and 1 for the training set.
- `xt = ...` — a larger held-out set, drawn the same way.
- `rng.normal(0.5, 0.5, 200)` — 200 held-out examples of class 1.
- `yt = ...` — labels for the held-out set.
- `n = len(y)` — the number of training examples, 80.
- `def sigmoid(v)` — the function that turns a score into a probability.
- `return 1.0 / ...` — the logistic formula.
- `def per_example_grads(w, b)` — computes all 80 gradients at once.
- `p = ...` — predicted probability for every training example.
- `return ...` — each row is `(p - y)` times the pair `[x, 1]`.
- `def clip(G, C)` — the per-example cap, applied to every row.
- `lengths = ...` — the length of each gradient row.
- `scale = ...` — 1 for short rows, `C / length` for long ones.
- `return G * ...` — multiply each row by its own factor.
- `def dp_sgd(...)` — the training loop, with its five settings.
- `rng = ...` — a fresh random stream for this run.
- `w, b = 0.0, 0.0` — start from a model that knows nothing.
- `accs = []` — collects accuracy so the plot can show progress.
- `for t in range(steps)` — one noisy update per step.
- `idx = ...` — the sampled examples for this step.
- `G = ...` — their clipped gradients.
- `total = ...` — sum the clipped rows, then add Gaussian noise.
- `w -= ...` — divide by the batch size and step.
- `b -= ...` — the same step for the bias.
- `pred = ...` — threshold the score at zero.
- `accs.append(...)` — fraction correct on the held-out set.
- `return accs` — the whole history.
- `accs = dp_sgd(...)` — run once at `sigma = 2`.
- `for t in range(0, len(accs), 30)` — print every thirtieth step.
- `print(f"step ...")` — an f-string with the step and its accuracy.
- `print("final accuracy"...)` — the last value in the history.

## 3. Plot: accuracy against noise

Train once per noise level and score each model on the same held-out examples. The curve starts near 84%, the ceiling set by the overlapping classes, and slides toward chance as the noise grows. Five seeds are averaged so one lucky run cannot bend the curve.

In [ ]:
import matplotlib.pyplot as plt                      # plotting library, preinstalled on Colab

sigmas = np.array([0., 1., 2., 3., 4., 5., 6., 8., 10.])   # noise multipliers to try
curve = []                                           # mean accuracy for each sigma
for s in sigmas:                                     # one setting at a time
    runs = [dp_sgd(sigma=float(s), seed=k)[-20:] for k in range(5)]   # last 20 steps, 5 seeds
    curve.append(float(np.mean(runs)))               # average away the luck
plt.figure(figsize=(6, 3))                           # a wide, short figure
plt.plot(sigmas, curve, "o-", color="tab:blue")      # accuracy against noise
plt.axhline(0.5, color="grey", linestyle="--", label="chance")   # a coin flip
plt.ylim(0.4, 0.95)                                  # zoom on the interesting range
plt.xlabel("noise multiplier sigma")                 # x-axis label
plt.ylabel("accuracy on 400 held-out examples")      # y-axis label
plt.legend()                                         # explain the dashed line
plt.title("more noise buys privacy and costs accuracy")   # the takeaway
plt.show()                                           # render the figure

**Line by line**

- `import matplotlib.pyplot as plt` — loads the plotting library.
- `sigmas = ...` — the noise levels, from none to strong.
- `curve = []` — one average accuracy per level.
- `for s in sigmas` — train a model at each level.
- `runs = ...` — five seeds, keeping only the last 20 accuracies.
- `curve.append(...)` — average all 100 numbers into one point.
- `plt.figure(...)` — a fresh canvas.
- `plt.plot(...)` — the sigma-versus-accuracy curve.
- `plt.axhline(0.5, ...)` — the chance line, for scale.
- `plt.ylim(...)` — crop the empty space above and below.
- `plt.xlabel(...)` — names the horizontal axis.
- `plt.ylabel(...)` — names the vertical axis.
- `plt.legend()` — names the dashed line.
- `plt.title(...)` — the takeaway.
- `plt.show()` — draws the figure.

## 4. Interactive: the clipping norm on a slider

The clipping norm `C` sets both the largest allowed contribution and the size of the noise, which is `sigma * C`. Drag it and watch the training curve redraw; the title reports how many gradients the cap touches and the final accuracy.

In [ ]:
from ipywidgets import interact                       # sliders that rerun a function

def show(C=1.0):                                     # C comes from the slider
    history = dp_sgd(C=C, sigma=6.0, seed=0)         # one noisy run at this cap
    lengths = np.linalg.norm(per_example_grads(0.0, 0.0), axis=1)   # lengths at the start
    touched = float((lengths > C).mean())            # share of gradients the cap changes
    plt.figure(figsize=(6, 3))                       # fresh figure on every drag
    plt.plot(range(1, len(history) + 1), history, color="tab:teal")   # accuracy per step
    plt.ylim(0.4, 0.95)                              # fixed axis, so nothing jumps
    plt.xlabel("step")                               # x-axis label
    plt.ylabel("accuracy on the held-out set")       # y-axis label
    plt.title(f"C={C:.2f}: {touched:.0%} clipped, acc={np.mean(history[-20:]):.2f}")   # readout
    plt.show()                                       # draw it

interact(show, C=(0.05, 2.0, 0.05))                  # slider: clipping norm 0.05 to 2.0

**Line by line**

- `from ipywidgets import interact` — Colab ships the widgets library.
- `def show(C=1.0)` — the function attached to the slider.
- `history = ...` — train once at this clipping norm with `sigma = 6`.
- `lengths = ...` — gradient lengths at the untrained model.
- `touched = ...` — fraction of rows that clipping actually shortens.
- `plt.figure(...)` — a new canvas per slider position.
- `plt.plot(...)` — accuracy after every step.
- `plt.ylim(...)` — keep the axis fixed across slider positions.
- `plt.xlabel(...)` — names the horizontal axis.
- `plt.ylabel(...)` — names the vertical axis.
- `plt.title(...)` — live readout of the cap, the share clipped, the accuracy.
- `plt.show()` — draws the frame.
- `interact(show, C=...)` — the slider, its range and step.

## 5. When it breaks: no noise, one example owns the update

Add one extreme example and remove the noise. Its gradient is dozens of times longer than the others, so the sum is mostly that one example and every other example is outvoted. The bars compare each example's share of the update with and without clipping.

In [ ]:
import numpy as np                                   # reload the array library
import matplotlib.pyplot as plt                      # reload the plotting library

x_out = np.concatenate([x, [20.0]])                  # the 80 examples plus one extreme value
y_out = np.concatenate([y, [0.0]])                   # the extreme example is labelled class 0
w0, b0 = 2.0, 0.0                                    # a model that already fits the 80 clean ones
p_out = sigmoid(w0 * x_out + b0)                     # its predictions, outlier included
G_out = (p_out - y_out)[:, None] * np.stack([x_out, np.ones(len(y_out))], axis=1)   # gradients
lengths = np.linalg.norm(G_out, axis=1)              # one length per example
raw = lengths / lengths.sum()                        # share of the unclipped sum
C = 1.0                                              # the clipping norm from section 1
cap = np.minimum(lengths, C)                         # each contribution after the cap
clipped = cap / cap.sum()                            # share of the clipped sum

print("outlier gradient length:", round(float(lengths[-1]), 2))   # about 20
print("median gradient length :", round(float(np.median(lengths)), 2))   # about 0.30
print("outlier share, no clipping:", f"{raw[-1]:.0%}")   # nearly half the update
print("outlier share, clipped   :", f"{clipped[-1]:.0%}")   # a few percent

plt.figure(figsize=(6, 3))                           # a wide, short figure
plt.bar(range(1, len(lengths) + 1), raw, color="tab:red", label="no clipping")   # raw shares
plt.bar(range(1, len(lengths) + 1), clipped, width=0.5, color="tab:blue", label="clipped")   # capped
plt.yscale("log")                                    # the shares span two orders of magnitude
plt.xlabel("example number (81 is the outlier)")     # x-axis label
plt.ylabel("share of the total update")              # y-axis label
plt.legend()                                         # name the two bar sets
plt.title("one bad example can own the update")      # the takeaway
plt.show()                                           # render the figure

**Line by line**

- `import numpy as np` — the array library, reloaded for this cell.
- `import matplotlib.pyplot as plt` — the plotting library.
- `x_out = ...` — the training features plus one value far outside the range.
- `y_out = ...` — the extra example is labelled class 0.
- `w0, b0 = 2.0, 0.0` — a model that already separates the 80 clean examples.
- `p_out = ...` — predictions for all 81 examples.
- `G_out = ...` — one gradient row per example, including the outlier.
- `lengths = ...` — how loudly each example speaks.
- `raw = ...` — each length as a fraction of the total.
- `C = 1.0` — the same cap as before.
- `cap = ...` — every contribution limited to `C`.
- `clipped = ...` — the shares after that limit, which add to one.
- `print("outlier gradient length"...)` — the outlier's length.
- `print("median gradient length"...)` — a typical example's length.
- `print("outlier share, no clipping"...)` — how much of the update it owns.
- `print("outlier share, clipped"...)` — the same share after the cap.
- `plt.figure(...)` — a fresh canvas.
- `plt.bar(...)` — raw shares, one bar per example.
- `plt.bar(...)` — clipped shares, drawn narrower so both are visible.
- `plt.yscale("log")` — the bars differ by a factor of a hundred.
- `plt.xlabel(...)` — names the horizontal axis.
- `plt.ylabel(...)` — names the vertical axis.
- `plt.legend()` — names red versus blue.
- `plt.title(...)` — the takeaway.
- `plt.show()` — draws the chart.

## Your turn

Set `sigma = 0.0` in section 2 and then in section 4: with no noise, what should the clipping slider change, and what does it actually change? Finally, predict the outlier's share in section 5 when `C` is raised to the outlier's own length.